# Proyecto #4: Pipeline de Pre-procesamiento, Entrenamiento e Inferencia

# Librerias 

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import FunctionTransformer

import joblib

# Asegurar carpetas de salida
os.makedirs("./models", exist_ok=True)
os.makedirs("./data/interim", exist_ok=True)

RSEED = 42
np.random.seed(RSEED)

# Datos

In [22]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\lalvarez\Product_Dev\repo-proyecto-final\data\interim\feature_exploration_scaled.csv"
)

df.head()

,ORDERDATE,CITY,PRODUCTLINE,STATUS,QUANTITYORDERED,PRICEEACH,SALES,CITY_TOP,CITY_TOP_Madrid,CITY_TOP_Manchester,...,STATUS_Shipped,SALES_LOG1P,PRICEEACH_LOG1P,QUANTITYORDERED_YJ,SALES_LOG1P_STD,PRICEEACH_LOG1P_STD,QUANTITYORDERED_YJ_STD,SALES_MM,PRICEEACH_MM,QUANTITYORDERED_MM
0,2003-01-06,Nashua,Vintage Cars,Shipped,30.0,100.000,5151.00,Other,0.0,0.0,...,1.0,8.547140,4.615121,10.779233,0.977215,0.744732,-0.502703,0.515174,1.000000,0.279486
1,2003-01-06,Nashua,Vintage Cars,Shipped,50.0,67.800,3390.00,Other,0.0,0.0,...,1.0,8.128880,4.231204,14.925497,0.161327,-0.604671,1.557932,0.302584,0.519159,0.838457
2,2003-01-06,Nashua,Vintage Cars,Shipped,22.0,86.510,1903.22,Other,0.0,0.0,...,1.0,7.551828,4.471753,8.805943,-0.964311,0.240819,-1.483401,0.123099,0.798554,0.055897
3,2003-01-06,Nashua,Vintage Cars,Shipped,49.0,34.470,1689.03,Other,0.0,0.0,...,1.0,7.432502,3.568687,14.736932,-1.197077,-2.933305,1.464218,0.097242,0.021444,0.810509
4,2003-01-09,Frankfurt,Vintage Cars,Shipped,45.0,33.034,1404.00,Other,0.0,0.0,...,1.0,7.247793,3.527360,13.966075,-1.557384,-3.078563,1.081113,0.062833,0.000000,0.698714


In [27]:
import numpy as np
import pandas as pd

# --- Definir target y features ---
y = df["SALES"]                  # variable objetivo
X = df.drop(columns=["SALES"])   # todas las demás como features

# --- Separar columnas numéricas y categóricas automáticamente ---
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols     = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numéricas:", numeric_cols)
print("Categóricas:", cat_cols)

Numéricas: ['QUANTITYORDERED', 'PRICEEACH', 'CITY_TOP_Madrid', 'CITY_TOP_Manchester', 'CITY_TOP_Melbourne', 'CITY_TOP_NYC', 'CITY_TOP_Nantes', 'CITY_TOP_New Bedford', 'CITY_TOP_Other', 'CITY_TOP_Paris', 'CITY_TOP_San Francisco', 'CITY_TOP_San Rafael', 'CITY_TOP_Singapore', 'PRODUCTLINE_Classic Cars', 'PRODUCTLINE_Motorcycles', 'PRODUCTLINE_Planes', 'PRODUCTLINE_Ships', 'PRODUCTLINE_Trains', 'PRODUCTLINE_Trucks and Buses', 'PRODUCTLINE_Vintage Cars', 'STATUS_Cancelled', 'STATUS_Disputed', 'STATUS_In Process', 'STATUS_On Hold', 'STATUS_Resolved', 'STATUS_Shipped', 'SALES_LOG1P', 'PRICEEACH_LOG1P', 'QUANTITYORDERED_YJ', 'SALES_LOG1P_STD', 'PRICEEACH_LOG1P_STD', 'QUANTITYORDERED_YJ_STD', 'SALES_MM', 'PRICEEACH_MM', 'QUANTITYORDERED_MM']
Categóricas: ['ORDERDATE', 'CITY', 'PRODUCTLINE', 'STATUS', 'CITY_TOP']


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- Bloque numérico: imputación + escalado ---
num_imputer = SimpleImputer(strategy="median")
num_scaler  = StandardScaler()

numeric_block = Pipeline(steps=[
    ("imputer", num_imputer),
    ("scaler", num_scaler),
])

# --- Bloque categórico: imputación por moda + OneHotEncoder ---
# IMPORTANTE: en sklearn 1.6+ se usa sparse_output, NO sparse
cat_imputer = SimpleImputer(strategy="most_frequent")
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

categorical_block = Pipeline(steps=[
    ("imputer", cat_imputer),
    ("encoder", ohe),
])

# --- ColumnTransformer que aplica cada bloque a sus columnas ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_block, numeric_cols),
        ("cat", categorical_block, cat_cols),
    ],
    remainder="drop",
)

# --- Pipeline completo de features (solo preprocesamiento) ---
feature_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor)
])

In [29]:
X_trans = feature_pipeline.fit_transform(X, y)

print("Shape original X:", X.shape)
print("Shape transformado X_trans:", X_trans.shape)

Shape original X: (2823, 40)
Shape transformado X_trans: (2823, 384)


In [30]:
import joblib
import os

os.makedirs("./models", exist_ok=True)

out_path = "./models/feature_pipeline.pkl"
joblib.dump(feature_pipeline, out_path)

print(" Pipeline de ingeniería de características guardado en:", out_path)

 Pipeline de ingeniería de características guardado en: ./models/feature_pipeline.pkl
